In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import insert_log
from pyspark.sql.functions import current_timestamp, lit, to_date, col
from datetime import datetime, timedelta
from lib.aws_s3 import warehouse_dir

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("02 - Date Production")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "date"
schema_name = "prd"
table_name = f"dim_{table}"
table_full_name = f"{schema_name}.{table_name}"

print("****************************************")
print("02 - Date Production")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print("")

In [ ]:
# Generate dates
def generate_dates(start=1900, end=2100):
    data = []
    for yr in range(start, end+1):
        #inner loop to print every day in the year
        curr_dt = datetime(yr, 1, 1)
        end_dt = datetime(yr, 12, 31)
        while curr_dt <= end_dt:
            data.append([
                curr_dt.strftime("%Y-%m-%d"),
                int(curr_dt.strftime("%d")),
                int(curr_dt.strftime("%m")),
                int(curr_dt.strftime("%Y")),
                (curr_dt.month - 1) // 3 + 1,
                curr_dt.strftime("%A"),
                curr_dt.weekday() in (5, 6)
            ])
            curr_dt += timedelta(days=1)
    return data

cols = ["date", "day", "month", "year", "quarter", "day_of_week", "is_weekend"]
data = generate_dates()

In [ ]:
# Create raw dataframe using data and cols
df_raw = spark.createDataFrame(data=data, schema=cols)

In [ ]:
print(f"df_raw: {df_raw.count()} records\n")
df_raw.printSchema()

In [ ]:
# Format data type and add audit column
df_prd = (
    df_raw
    .withColumn("insert_ts", lit(now_ts))
    .withColumn("date", to_date('date', "yyyy-MM-dd"))
    .withColumn("day", col('day').cast('int'))
    .withColumn("month", col('month').cast('int'))
    .withColumn("year", col('year').cast('int'))
    .withColumn("quarter", col('quarter').cast('int'))
)

In [ ]:
print(f"df_prd: {df_prd.count()} records\n")
df_prd.printSchema()

In [ ]:
# Load data to prd.dim_date
(
    df_prd
    .coalesce(4)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_full_name)
)

In [ ]:
print(f"{df_prd.count()} records written to {table_full_name} (mode=overwrite)\n")

In [ ]:
# Add entry to prd.job_control
if insert_log(spark, schema_name, table_name, now_ts, run_date):
    print(f"{schema_name}.{table_name} entry added to prd.job_control")

In [ ]:
# Review added entry
spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)

In [ ]:
# stop spark s ession
#spark.stop()